# ASAP-AES Exploratory Data Analysis

This notebook performs exploratory data analysis on the ASAP-AES (Automated Student Assessment Prize) dataset from Kaggle.

**Objectives:**
- Load and inspect the ASAP-AES Excel files
- Analyze dataset structure and statistics (essay counts, score ranges, etc.)
- Compute inter-rater agreement (Quadratic Weighted Kappa)
- Visualize score distributions across essay sets
- Identify outliers and data quality issues
- Generate summary statistics table

**Prerequisites:**
- ASAP-AES Excel files in `../data/` or specify path below
- Python packages: pandas, numpy, matplotlib, seaborn, scikit-learn

In [ ]:
# Import libraries
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add src to path for importing our modules
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))

# Import our custom modules (will work after data is available)
# from data_loader import ASAPDataLoader
# from metrics import compute_qwk, ordinal_from_continuous

# Set up plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

In [ ]:
# Configuration: Specify data paths
# ===================================

# DATA SOURCE: ASAP-AES Excel files from Unified_Datasets folder
# Path to ASAP-AES data directory (containing .xlsx files)
# This consolidates all training data in one location
DATA_DIR = Path('../../Unified_Datasets')  # Default: SwiftGrade-Models/Unified_Datasets/

# If your ASAP-AES files are located elsewhere, update the path:
# DATA_DIR = Path('C:/path/to/Unified_Datasets')

# Check if data directory exists
if not DATA_DIR.exists():
    print(f"⚠️  Data directory not found: {DATA_DIR}")
    print("Please ensure ASAP-AES Excel files are copied to Unified_Datasets folder")
    print("Or update DATA_DIR to point to your data location")
else:
    excel_files = list(DATA_DIR.glob('*.xlsx'))
    print(f"✓ Found {len(excel_files)} Excel file(s) in {DATA_DIR}")
    if excel_files:
        for f in excel_files[:5]:
            print(f"  - {f.name}")
        if len(excel_files) > 5:
            print(f"  ... and {len(excel_files) - 5} more")

In [ ]:
# Load ASAP-AES Data
# ==================

# ASAP-AES essay sets metadata
# (Standard Kaggle ASAP-AES structure)
ESSAY_SET_METADATA = {
    1: {"name": "Persuasive Essays", "min_score": 2, "max_score": 12},
    2: {"name": "Persuasive Essays", "min_score": 1, "max_score": 6},
    3: {"name": "Persuasive Essays", "min_score": 0, "max_score": 3},
    4: {"name": "Persuasive Essays", "min_score": 0, "max_score": 3},
    5: {"name": "Argumentative Essays", "min_score": 0, "max_score": 4},
    6: {"name": "Argumentative Essays", "min_score": 0, "max_score": 4},
    7: {"name": "Argumentative Essays", "min_score": 0, "max_score": 30},
    8: {"name": "Argumentative Essays", "min_score": 0, "max_score": 60},
}

# Function to load all ASAP-AES Excel files
def load_asap_aes_data(data_dir):
    """
    Load all ASAP-AES essay files from directory.
    Expected format: columns = ['essay_id', 'set_id', 'essay_text', 'score1', 'score2', ...]
    """
    data_by_set = {}
    excel_files = sorted(data_dir.glob('*.xlsx'))
    
    if not excel_files:
        print("❌ No Excel files found in data directory")
        return data_by_set
    
    for file_path in excel_files:
        try:
            df = pd.read_excel(file_path)
            set_id = df['essay_set'].unique()[0] if 'essay_set' in df.columns else None
            
            if set_id is None:
                # Try alternative column names
                if 'set_id' in df.columns:
                    set_id = df['set_id'].unique()[0]
                elif 'essay_set' in df.columns:
                    set_id = df['essay_set'].unique()[0]
            
            data_by_set[set_id] = df
            print(f"✓ Loaded essay set {set_id}: {len(df)} essays from {file_path.name}")
        except Exception as e:
            print(f"❌ Failed to load {file_path.name}: {str(e)}")
    
    return data_by_set

# Load data
data_by_set = load_asap_aes_data(DATA_DIR)

if data_by_set:
    print(f"\n✓ Total essay sets loaded: {len(data_by_set)}")
else:
    print("\n⚠️  No data loaded. Please check data directory and file format.")

In [ ]:
# Inspect Dataset Structure
# ==========================

if data_by_set:
    print("=" * 80)
    print("DATASET INSPECTION")
    print("=" * 80)
    
    for set_id, df in sorted(data_by_set.items()):
        print(f"\n📋 Essay Set {set_id} ({ESSAY_SET_METADATA[set_id]['name']})")
        print(f"   Essays: {len(df)}")
        print(f"   Columns: {list(df.columns)}")
        print(f"   Data types:\n{df.dtypes}")
        print(f"   Missing values: {df.isnull().sum().sum()}")
        
        # Display first essay
        if 'essay' in df.columns:
            essay_col = 'essay'
        elif 'text' in df.columns:
            essay_col = 'text'
        else:
            # Find the most text-like column
            essay_col = [col for col in df.columns if df[col].dtype == 'object'][0]
        
        first_essay = df[essay_col].iloc[0]
        print(f"   First essay (preview): {first_essay[:100]}...")
else:
    print("⚠️  No data available for inspection. Please load data first.")

In [ ]:
# Score Statistics and Distributions
# ===================================

if data_by_set:
    print("=" * 80)
    print("SCORE STATISTICS")
    print("=" * 80)
    
    score_stats = []
    
    for set_id, df in sorted(data_by_set.items()):
        # Find score columns (typically 'score', 'score1', 'score2', etc.)
        score_cols = [col for col in df.columns if 'score' in col.lower()]
        
        if score_cols:
            # Use the first score column (usually the gold/average score)
            primary_score_col = score_cols[0]
            scores = df[primary_score_col]
            
            stats = {
                'Essay Set': set_id,
                'Type': ESSAY_SET_METADATA[set_id]['name'],
                'Essays': len(df),
                'Mean Score': f"{scores.mean():.2f}",
                'Median Score': f"{scores.median():.2f}",
                'Std Dev': f"{scores.std():.2f}",
                'Min': f"{scores.min():.0f}",
                'Max': f"{scores.max():.0f}",
                'Range': f"{ESSAY_SET_METADATA[set_id]['min_score']}-{ESSAY_SET_METADATA[set_id]['max_score']}"
            }
            score_stats.append(stats)
            
            print(f"\n📊 Essay Set {set_id}:")
            for key, val in stats.items():
                if key != 'Essay Set':
                    print(f"   {key}: {val}")
    
    # Create summary DataFrame
    if score_stats:
        stats_df = pd.DataFrame(score_stats)
        print("\n" + "=" * 80)
        print("SUMMARY TABLE")
        print("=" * 80)
        print(stats_df.to_string(index=False))
else:
    print("⚠️  No data available for statistics. Please load data first.")

In [ ]:
# Visualize Score Distributions
# =============================

if data_by_set and len(data_by_set) > 0:
    n_sets = len(data_by_set)
    fig, axes = plt.subplots(n_sets, 1, figsize=(12, 4 * n_sets))
    
    if n_sets == 1:
        axes = [axes]
    
    for idx, (set_id, df) in enumerate(sorted(data_by_set.items())):
        ax = axes[idx]
        
        # Find score column
        score_cols = [col for col in df.columns if 'score' in col.lower()]
        if score_cols:
            primary_score_col = score_cols[0]
            scores = df[primary_score_col]
            
            # Histogram with KDE
            ax.hist(scores, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
            ax.set_xlabel('Score', fontsize=11)
            ax.set_ylabel('Frequency', fontsize=11)
            ax.set_title(f'Essay Set {set_id}: Score Distribution (n={len(df)})', fontsize=12, fontweight='bold')
            ax.grid(alpha=0.3)
            
            # Add statistics text
            stats_text = f"Mean: {scores.mean():.2f} | Median: {scores.median():.2f} | Std: {scores.std():.2f}"
            ax.text(0.98, 0.97, stats_text, transform=ax.transAxes, 
                   verticalalignment='top', horizontalalignment='right',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                   fontsize=9)
    
    plt.tight_layout()
    plt.show()
    print("✓ Score distributions plotted")
else:
    print("⚠️  No data available for visualization.")

In [ ]:
# Inter-Rater Agreement Analysis (QWK)
# =====================================

from sklearn.metrics import cohen_kappa_score

def compute_qwk(y_true, y_pred):
    """Compute Quadratic Weighted Kappa between two raters."""
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

if data_by_set:
    print("=" * 80)
    print("INTER-RATER AGREEMENT (QWK - Quadratic Weighted Kappa)")
    print("=" * 80)
    print("Note: QWK measures agreement between human raters/scorers")
    print("Range: -1 to 1 (1 = perfect agreement, 0 = random, -1 = disagreement)")
    print()
    
    qwk_results = []
    
    for set_id, df in sorted(data_by_set.items()):
        # Look for multiple score columns (score1, score2) indicating multiple raters
        score_cols = sorted([col for col in df.columns if 'score' in col.lower()])
        
        if len(score_cols) >= 2:
            # Compute QWK between first two raters
            score1 = df[score_cols[0]].dropna().astype(int)
            score2 = df[score_cols[1]].dropna().astype(int)
            
            # Align lengths
            min_len = min(len(score1), len(score2))
            score1 = score1.iloc[:min_len]
            score2 = score2.iloc[:min_len]
            
            if len(score1) > 0 and len(score2) > 0:
                qwk = compute_qwk(score1, score2)
                qwk_results.append({
                    'Essay Set': set_id,
                    'QWK (Rater 1 vs 2)': f"{qwk:.4f}",
                    'Essays Compared': len(score1)
                })
                print(f"✓ Set {set_id}: QWK = {qwk:.4f} (n={len(score1)})")
        else:
            print(f"⚠️  Set {set_id}: Only one rater/score column found (cannot compute QWK)")
    
    if qwk_results:
        print("\n" + "=" * 80)
        qwk_df = pd.DataFrame(qwk_results)
        print(qwk_df.to_string(index=False))
else:
    print("⚠️  No data available for inter-rater analysis.")

In [ ]:
# Essay Length Analysis
# ======================

if data_by_set:
    print("=" * 80)
    print("ESSAY LENGTH ANALYSIS")
    print("=" * 80)
    
    length_stats = []
    
    for set_id, df in sorted(data_by_set.items()):
        # Find essay text column
        essay_cols = [col for col in df.columns if col.lower() in ['essay', 'text', 'essay_text']]
        if not essay_cols:
            # Fallback: find object columns that look like text
            essay_cols = [col for col in df.columns if df[col].dtype == 'object' and col not in ['essay_set', 'set_id']]
        
        if essay_cols:
            essay_col = essay_cols[0]
            
            # Compute essay lengths
            essay_lengths = df[essay_col].fillna('').str.split().str.len()
            
            length_stat = {
                'Essay Set': set_id,
                'Avg Length (words)': f"{essay_lengths.mean():.1f}",
                'Median Length': f"{essay_lengths.median():.0f}",
                'Min Length': f"{essay_lengths.min():.0f}",
                'Max Length': f"{essay_lengths.max():.0f}",
                'Std Dev': f"{essay_lengths.std():.1f}"
            }
            length_stats.append(length_stat)
            
            print(f"\n📝 Essay Set {set_id}:")
            for key, val in length_stat.items():
                if key != 'Essay Set':
                    print(f"   {key}: {val}")
    
    if length_stats:
        length_df = pd.DataFrame(length_stats)
        print("\n" + "=" * 80)
        print(length_df.to_string(index=False))
else:
    print("⚠️  No data available for length analysis.")

In [ ]:
# Next Steps & Recommendations
# =============================

print("=" * 80)
print("NEXT STEPS")
print("=" * 80)

if data_by_set:
    print("\n✓ Data exploration complete!")
    print("\nRecommendations:")
    print("  1. Run 'Train_Essay_Scoring_Model.ipynb' to:")
    print("     - Extract linguistic features for all essays")
    print("     - Train XGBoost regressors per essay set")
    print("     - Optimize models for QWK metric (not MSE)")
    print("     - Save trained models to ../models/")
    print()
    print("  2. Use 'Evaluate_Essays.ipynb' to:")
    print("     - Score new essays interactively")
    print("     - Generate personalized feedback")
    print("     - Export results to CSV")
    print()
    print("📌 Key Insights:")
    print(f"   - Total essay sets available: {len(data_by_set)}")
    print(f"   - Each set has unique score range and rubric")
    print(f"   - Different essay lengths per set (affects feature importance)")
    print(f"   - Use per-set models for better generalization")
else:
    print("\n⚠️  Data not loaded. Please:")
    print("  1. Copy ASAP-AES Excel files to ../data/ directory")
    print("  2. Or update DATA_DIR variable to point to your data location")
    print("  3. Re-run all cells above")

print("\n" + "=" * 80)